[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C3/notebooks/praporek.ipynb)

# Červený praporek: najděte chybu v odhadu složení krve

Kolega odhadoval složení vzorku krve ze čtyř typů buněk podle exprese šesti
genů. Exprese směsi je vážený průměr expresí čistých typů (matice $A$),
váhami jsou podíly buněk $\mathbf x$:

$$
\begin{aligned}
\text{minimize}\quad & f(\mathbf x) = \|A\mathbf x-\mathbf b\|_2^2\\
\text{subject to}\quad & \mathbf x \ge \mathbf 0,\quad x_1+x_2+x_3+x_4 = 1
\end{aligned}
$$

Poslal výsledek: **T 41,1 %, NK 0 %, monocyty 13,4 %, neutrofily 45,5 %.
Nic záporného, součet 100 % ✓.** Nejmenší čtverce mu daly NK buňky záporné,
tak je nastavil na nulu a zbytek přenormoval. Je to dobře?

## Postup

1. spustit kód kolegy: splňuje výsledek zadání?
2. je to ale **nejlepší** odhad, jaký data dovolují? Rozhodnout dvěma cestami:
   vyřešit správnou úlohu a porovnat $f$ *(díra 1)*, nebo bez solveru
   z gradientu *(díra 2)*.

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/cerveny_praporek_orezani.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C3/kod/cerveny_praporek_orezani.py), [`kod/praporek_orezani.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C3/kod/praporek_orezani.py).*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TYPY = ["T-lymfocyty", "NK buňky", "monocyty", "neutrofily"]
# řádky geny CD3E, NKG7, GNLY, CD14, FCGR3B, LYZ; sloupce typy buněk
A = np.array([[100, 10, 1, 1],
              [40, 120, 2, 2],
              [20, 80, 1, 1],
              [2, 2, 150, 10],
              [1, 20, 5, 200],
              [5, 5, 300, 120]], dtype=float)
b = np.array([45, 10, 2, 30, 100, 105], dtype=float)     # exprese ve směsi


def f(x):
    return float(np.sum((A @ x - b) ** 2))

## Řešení kolegy

In [ ]:
x_ls = np.linalg.lstsq(A, b, rcond=None)[0]
x_kolega = np.clip(x_ls, 0, None)          # záporný podíl nedává smysl -> 0
x_kolega = x_kolega / x_kolega.sum()       # a ať to dá dohromady 100 %

for typ, podil in zip(TYPY, x_kolega):
    print(f"{typ:12s} {100 * podil:5.1f} %")
print(f"chyba fitu f = {f(x_kolega):.1f}")
assert np.all(x_kolega >= 0) and abs(x_kolega.sum() - 1) < 1e-9
print("kontrola: nic záporného, součet 100 % ✓  ->  hotovo")

## Cesta 1: správný model *(díra 1)*

Omezení $\mathbf x \ge \mathbf 0$ a součet 1 dejte do modelu v CVXPY.
Přípustný bod nemůže mít menší $f$ než optimum. Jak velký je rozdíl?

In [ ]:
# TODO 1: správný model: stejná f, omezení patří DO modelu.
x = cp.Variable(4)
nezapornost = ...   # podíly nezáporné
soucet = ...        # podíly dávají dohromady 1
omezeni = [nezapornost, soucet]
uloha = ...         # cp.Problem(cp.Minimize(...), omezeni)

In [ ]:
uloha.solve(solver=cp.CLARABEL)
x_opt = x.value
assert x_opt.min() >= -1e-9 and abs(x_opt.sum() - 1) < 1e-6, "optimum nesplňuje zadání"
print(f"kolega:  {np.round(100 * x_kolega, 1)} %,  f = {f(x_kolega):.1f}")
print(f"optimum: {np.round(100 * x_opt, 1)} %,  f = {f(x_opt):.1f}")
print(f"kolegova chyba je {f(x_kolega) / f(x_opt):.2f}krát větší")
assert abs(f(x_opt) - 138.57) < 0.01 and abs(f(x_kolega) - 266.17) < 0.01

## Cesta 2: test bez solveru *(díra 2)*

**Úvaha o přelévání.** Když přelijete kousek $\Delta$ z typu $i$ do typu $j$
(součet zůstane 100 %), změní se $f$ zhruba
o $(\partial f/\partial x_j - \partial f/\partial x_i)\,\Delta$. Gradient je
$\nabla f = 2A^\top(A\mathbf x-\mathbf b)$. Má u kolegy na nenulových podílech
**stejnou hodnotu**?

<details><summary><b>Nápověda</b> (otevřít, když se zaseknete)</summary>

Liší-li se složky gradientu na nenulových podílech, vždy se vyplatí přelít
z větší do menší, takže bod není optimum. Obecně totéž říká KKT stacionarita
$\nabla f + \lambda\mathbf 1 - \boldsymbol\mu = \mathbf 0$,
$\boldsymbol\mu \ge \mathbf 0$, $\mu_i x_i = 0$; $\lambda$ patří k součtu,
$\mu_i$ k nezápornosti. Nezápornost je $g_i = -x_i \le 0$, proto $-\boldsymbol\mu$.
Kde je $x_i > 0$, je $\mu_i = 0$, takže $\partial f/\partial x_i = -\lambda$:
na všech nenulových podílech stejné číslo. Na nulovém podílu smí být
gradient jen větší.

</details>

In [ ]:
# TODO 2: gradient f(x) = ||Ax - b||^2, nápověda: 2 * A.T @ (...)
def grad(x):
    return ...

In [ ]:
g_kol, g_opt = grad(x_kolega), grad(x_opt)
rozdil_kol = np.ptp(g_kol[x_kolega > 1e-6])   # max - min jen na nenulových podílech
rozdil_opt = np.ptp(g_opt[x_opt > 1e-6])
print(f"kolega   grad f = {np.round(g_kol)}, rozdíl na nenulových {rozdil_kol:.0f}")
print(f"optimum  grad f = {np.round(g_opt)}, rozdíl na nenulových {rozdil_opt:.0f}")
print(f"CVXPY: lambda = {soucet.dual_value:.1f}, mu = {np.round(nezapornost.dual_value, 1)}")
assert rozdil_opt < 1 and rozdil_kol > 6000

poz = np.arange(4)
plt.bar(poz - 0.2, g_kol, 0.4, label="kolega")
plt.bar(poz + 0.2, g_opt, 0.4, label="optimum")
plt.axhline(-soucet.dual_value, ls="--", color="gray", label=r"$-\lambda$")
plt.xticks(poz, TYPY)
plt.ylabel("složka gradientu ∇f")
plt.legend()
plt.show()

### Vaše věta

*(dvojklikem do téhle buňky se do ní dá psát; jedna věta, s číslem)*

**TODO:** Výsledek kolegy splňuje zadání, ale není optimální, protože …

## Poučení

Přípustné ≠ optimální. Omezení patří do modelu, ne do „opravy“ po výpočtu.
Neoptimalitu řešení kolegy prokáže jediný řádek: složky gradientu na
nenulových podílech se liší, takže přelitím jde $f$ snížit. Optimalitu bodu
dokazuje až celé KKT včetně nulových složek: tam musí být gradient aspoň
$-\lambda$, tedy $\mu_i \ge 0$.